# Bioyologic · Clasificador de especie por colonia (Google Colab)

Segunda etapa de Bioyologic: el **detector** ya encuentra y cuenta las colonias; este notebook entrena un **clasificador** (YOLO11-cls) que mira el recorte de cada colonia y dice su especie. La especie de la placa se decide por votación entre sus colonias.

Datos: los mismos dos datasets públicos (Makrai *et al.* 2023 y el dataset clínico de Zenodo), unificados por nombre de especie: **37 especies**.

**Cómo usarlo**: `Entorno de ejecución → Cambiar tipo → GPU T4` y `Ejecutar todo` (~1,5 h). Al final se descarga `bioyologic_species.zip` (modelo + configuración): descomprímelo en `models/`.

**Advertencia**: en estos datasets cada placa tiene una sola especie y cada especie se cultivó en su propio medio (sangre, MacConkey, etc.). El clasificador puede apoyarse en el color del agar además de en la colonia. Con fotos de otro laboratorio, la especie es orientativa y debe confirmarse con pruebas microbiológicas.


In [ ]:
# 1. GPU y dependencias
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ Sin GPU: activa T4 en 'Cambiar tipo de entorno de ejecución'"
%pip install -q ultralytics
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# 2. Configuración
from pathlib import Path

WORK = Path("/content")
RAW = WORK / "raw"
DATASET = WORK / "crops"                  # dataset de clasificación: split/especie/recorte.jpg

USE_MAKRAI = True
USE_CLINICAL = True

PAD = 0.30            # margen alrededor de la caja (se ve un poco de agar, como en el servidor)
CROP_SIZE = 160       # lado del recorte guardado (px)
MAX_PER_PLATE = 80    # recortes por placa como máximo (equilibra placas muy pobladas)
MIN_BOX = 12          # cajas más pequeñas (px) se descartan
SPLIT = (0.8, 0.1, 0.1)
SEED = 42

MODEL = "yolo11s-cls.pt"
IMGSZ = 160
EPOCHS = 40
TIME_HOURS = 1.0
BATCH = 256

# código de cada dataset -> especie (tablas 1 de ambos artículos)
SPECIES_MAP = {"makrai:sp01": {"species": "Actinobacillus equuli", "gram": "-", "agar": "blood"}, "makrai:sp02": {"species": "Actinobacillus pleuropneumoniae", "gram": "-", "agar": "chocolate"}, "makrai:sp03": {"species": "Aeromonas hydrophila", "gram": "-", "agar": "blood"}, "makrai:sp04": {"species": "Bacillus cereus", "gram": "+", "agar": "blood"}, "makrai:sp05": {"species": "Bibersteinia trehalosi", "gram": "-", "agar": "blood"}, "makrai:sp06": {"species": "Bordetella bronchiseptica", "gram": "-", "agar": "blood"}, "makrai:sp07": {"species": "Brucella ovis", "gram": "-", "agar": "blood"}, "makrai:sp08": {"species": "Clostridium perfringens", "gram": "+", "agar": "blood"}, "makrai:sp09": {"species": "Corynebacterium pseudotuberculosis", "gram": "+", "agar": "blood"}, "makrai:sp10": {"species": "Erysipelothrix rhusiopathiae", "gram": "+", "agar": "blood"}, "makrai:sp11": {"species": "Escherichia coli", "gram": "-", "agar": "nutrient"}, "makrai:sp12": {"species": "Glaesserella parasuis", "gram": "-", "agar": "chocolate"}, "makrai:sp13": {"species": "Klebsiella pneumoniae", "gram": "-", "agar": "blood, nutrient"}, "makrai:sp14": {"species": "Listeria monocytogenes", "gram": "+", "agar": "blood"}, "makrai:sp15": {"species": "Paenibacillus larvae", "gram": "+", "agar": "blood"}, "makrai:sp16": {"species": "Pasteurella multocida", "gram": "-", "agar": "blood"}, "makrai:sp17": {"species": "Proteus mirabilis", "gram": "-", "agar": "MacConkey"}, "makrai:sp18": {"species": "Pseudomonas aeruginosa", "gram": "-", "agar": "blood"}, "makrai:sp19": {"species": "Rhodococcus equi", "gram": "+", "agar": "blood"}, "makrai:sp20": {"species": "Salmonella enterica", "gram": "-", "agar": "nutrient"}, "makrai:sp21": {"species": "Staphylococcus aureus", "gram": "+", "agar": "blood"}, "makrai:sp22": {"species": "Staphylococcus hyicus", "gram": "+", "agar": "blood"}, "makrai:sp23": {"species": "Streptococcus agalactiae", "gram": "+", "agar": "blood"}, "makrai:sp24": {"species": "Trueperella pyogenes", "gram": "+", "agar": "blood"}, "clinical:aba": {"species": "Acinetobacter baumannii", "gram": "-", "agar": "MacConkey"}, "clinical:eco": {"species": "Escherichia coli", "gram": "-", "agar": "blood"}, "clinical:kpn": {"species": "Klebsiella pneumoniae", "gram": "-", "agar": "MacConkey"}, "clinical:sau": {"species": "Staphylococcus aureus", "gram": "+", "agar": "blood"}, "clinical:ses": {"species": "Salmonella enterica", "gram": "-", "agar": "Salmonella-Shigella"}, "clinical:sma": {"species": "Serratia marcescens", "gram": "-", "agar": "MacConkey"}, "clinical:sep": {"species": "Staphylococcus epidermidis", "gram": "+", "agar": "blood"}, "clinical:efa": {"species": "Enterococcus faecium", "gram": "+", "agar": "blood"}, "clinical:stm": {"species": "Stenotrophomonas maltophilia", "gram": "-", "agar": "MacConkey"}, "clinical:eca": {"species": "Enterococcus casseliflavus", "gram": "+", "agar": "blood"}, "clinical:enc": {"species": "Enterobacter cloacae", "gram": "-", "agar": "MacConkey"}, "clinical:cst": {"species": "Corynebacterium striatum", "gram": "+", "agar": "blood"}, "clinical:mmo": {"species": "Morganella morganii", "gram": "-", "agar": "MacConkey"}, "clinical:bce": {"species": "Bacillus cereus", "gram": "+", "agar": "blood"}, "clinical:spy": {"species": "Streptococcus pyogenes", "gram": "+", "agar": "blood"}, "clinical:sag": {"species": "Streptococcus agalactiae", "gram": "+", "agar": "blood"}, "clinical:ppu": {"species": "Pseudomonas putida", "gram": "-", "agar": "blood"}, "clinical:spn": {"species": "Streptococcus pneumoniae", "gram": "+", "agar": "blood"}, "clinical:bcp": {"species": "Burkholderia cepacia", "gram": "-", "agar": "MacConkey"}}

In [ ]:
# 3. Descarga de los datasets
import json, struct, urllib.request, zipfile, zlib
from concurrent.futures import ThreadPoolExecutor

def http_range(url, start, end):
    req = urllib.request.Request(url, headers={"Range": f"bytes={start}-{end}"})
    return urllib.request.urlopen(req, timeout=300).read()

def remote_zip_entries(url):
    """Índice de un ZIP remoto (también ZIP64) leyendo solo su directorio central."""
    size = int(urllib.request.urlopen(urllib.request.Request(url, method="HEAD")).headers["Content-Length"])
    tail = http_range(url, max(0, size - 65558), size - 1)
    i = tail.rfind(b"PK\x05\x06")
    cd_size, cd_off = struct.unpack("<II", tail[i + 12:i + 20])
    if cd_off == 0xFFFFFFFF or cd_size == 0xFFFFFFFF:
        j = tail.rfind(b"PK\x06\x06")
        cd_size, cd_off = struct.unpack("<QQ", tail[j + 40:j + 56])
    cd = http_range(url, cd_off, cd_off + cd_size - 1)
    entries, p = {}, 0
    while cd[p:p + 4] == b"PK\x01\x02":
        method = struct.unpack("<H", cd[p + 10:p + 12])[0]
        csize, usize = struct.unpack("<II", cd[p + 20:p + 28])
        n, e, c = struct.unpack("<HHH", cd[p + 28:p + 34])
        off = struct.unpack("<I", cd[p + 42:p + 46])[0]
        name = cd[p + 46:p + 46 + n].decode("utf-8", "replace")
        extra, q = cd[p + 46 + n:p + 46 + n + e], 0
        while q + 4 <= len(extra):
            hid, hlen = struct.unpack("<HH", extra[q:q + 4])
            if hid == 1:
                vals, k = extra[q + 4:q + 4 + hlen], 0
                if usize == 0xFFFFFFFF: usize = struct.unpack("<Q", vals[k:k + 8])[0]; k += 8
                if csize == 0xFFFFFFFF: csize = struct.unpack("<Q", vals[k:k + 8])[0]; k += 8
                if off == 0xFFFFFFFF: off = struct.unpack("<Q", vals[k:k + 8])[0]; k += 8
            q += 4 + hlen
        entries[name] = (method, csize, off)
        p += 46 + n + e + c
    return entries

def remote_zip_read(url, entries, name):
    method, csize, off = entries[name]
    n, e = struct.unpack("<HH", http_range(url, off, off + 29)[26:30])
    data = http_range(url, off + 30 + n + e, off + 30 + n + e + csize - 1)
    return data if method == 0 else zlib.decompress(data, -15)

def download(url, dest, size=None, parts=1):
    """Descarga a disco; con parts > 1, en trozos paralelos (Zenodo limita ~1 MB/s por conexión)."""
    if dest.exists() and (size is None or dest.stat().st_size == size):
        return
    tmp = dest.with_suffix(dest.suffix + ".part")
    if parts == 1:
        with urllib.request.urlopen(url, timeout=300) as r, open(tmp, "wb") as f:
            while chunk := r.read(8 << 20):
                f.write(chunk)
    else:
        total = int(urllib.request.urlopen(urllib.request.Request(url, method="HEAD")).headers["Content-Length"])
        with open(tmp, "wb") as f:
            f.truncate(total)
        step = -(-total // parts)
        def piece(i):
            start, end = i * step, min(total, (i + 1) * step) - 1
            req = urllib.request.Request(url, headers={"Range": f"bytes={start}-{end}"})
            with urllib.request.urlopen(req, timeout=300) as r, open(tmp, "r+b") as f:
                f.seek(start)
                while chunk := r.read(8 << 20):
                    f.write(chunk)
        with ThreadPoolExecutor(parts) as pool:
            list(pool.map(piece, range(parts)))
    tmp.rename(dest)

SOURCES = []   # (nombre, coco.json, carpeta de imágenes)

if USE_MAKRAI:
    d = RAW / "makrai"
    d.mkdir(parents=True, exist_ok=True)
    meta = json.load(urllib.request.urlopen("https://api.figshare.com/v2/articles/22022540"))
    files = [f for f in meta["files"] if f["name"].lower().endswith(".jpg") or f["name"] == "annot_COCO.json"]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda f: download(f["download_url"], d / f["name"], f["size"]), files))
    SOURCES.append(("makrai", d / "annot_COCO.json", d))
    print(f"Makrai: {len(files)} archivos")

if USE_CLINICAL:
    d = RAW / "clinical"
    d.mkdir(parents=True, exist_ok=True)
    base = "https://zenodo.org/api/records/18256864/files"
    # Las etiquetas (label.zip, 2,6 GB) incluyen JSON de LabelMe con la imagen embebida:
    # solo se extrae coco.json (~1 MB) leyendo ese fragmento del ZIP remoto.
    if not (d / "coco.json").exists():
        url = f"{base}/label.zip/content"
        (d / "coco.json").write_bytes(remote_zip_read(url, remote_zip_entries(url), "label/coco_json/coco.json"))
    if not (d / "images").exists():
        download(f"{base}/images.zip/content", d / "images.zip", parts=16)
        with zipfile.ZipFile(d / "images.zip") as z:
            z.extractall(d)
        (d / "images.zip").unlink()
    SOURCES.append(("clinical", d / "coco.json", d / "images"))
    print("Clinical: listo")

!du -sh {RAW}/*

In [ ]:
# 4. Recortes de colonias por especie (partición por placa dentro de cada especie)
import json, random, shutil
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import cv2
import numpy as np

def square_crop(img, box, pad=PAD):
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(x2 - x1, y2 - y1) * (1 + 2 * pad)
    h, w = img.shape[:2]
    a, b = int(max(0, cx - side / 2)), int(max(0, cy - side / 2))
    c, d = int(min(w, cx + side / 2)), int(min(h, cy + side / 2))
    crop = img[b:d, a:c]
    return cv2.resize(crop, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_AREA) if crop.size else None

plates = defaultdict(list)   # especie -> [(ruta, [cajas])]
for src_name, coco_path, img_dir in SOURCES:
    coco = json.loads(Path(coco_path).read_text())
    cats = {c["id"]: SPECIES_MAP[f"{src_name}:{c['name']}"]["species"] for c in coco["categories"]}
    index = {p.name: p for p in Path(img_dir).rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")}
    by_img = defaultdict(list)
    for a in coco["annotations"]:
        x, y, w, h = a["bbox"]
        if max(w, h) >= MIN_BOX:
            by_img[a["image_id"]].append((cats[a["category_id"]], (x, y, x + w, y + h)))
    for im in coco["images"]:
        path = index.get(Path(im["file_name"]).name)
        boxes = by_img.get(im["id"], [])
        if path is None or not boxes:
            continue
        sp = max(set(b[0] for b in boxes), key=lambda s: sum(1 for b in boxes if b[0] == s))
        plates[sp].append((path, [b[1] for b in boxes]))

rng = random.Random(SEED)
parts = {"train": [], "val": [], "test": []}
for sp, items in sorted(plates.items()):
    rng.shuffle(items)
    n = len(items)
    n_te = max(1, round(n * SPLIT[2])) if n >= 3 else 0
    n_va = max(1, round(n * SPLIT[1])) if n >= 3 else 0
    parts["test"] += [(sp, it) for it in items[:n_te]]
    parts["val"] += [(sp, it) for it in items[n_te:n_te + n_va]]
    parts["train"] += [(sp, it) for it in items[n_te + n_va:]]

shutil.rmtree(DATASET, ignore_errors=True)

def export(args):
    part, sp, (path, boxes) = args
    img = cv2.imread(str(path))
    if img is None:
        return 0
    if len(boxes) > MAX_PER_PLATE:
        boxes = random.Random(str(path)).sample(boxes, MAX_PER_PLATE)
    folder = DATASET / part / sp.replace(" ", "_")
    folder.mkdir(parents=True, exist_ok=True)
    n = 0
    for k, b in enumerate(boxes):
        crop = square_crop(img, b)
        if crop is not None:
            cv2.imwrite(str(folder / f"{path.stem}_{k}.jpg"), crop, [cv2.IMWRITE_JPEG_QUALITY, 92])
            n += 1
    return n

summary = {}
for part, items in parts.items():
    with ThreadPoolExecutor(4) as pool:
        counts = list(pool.map(export, [(part, sp, it) for sp, it in items]))
    summary[part] = {"placas": len(items), "recortes": sum(counts)}
TEST_PLATES = parts["test"]
CLASSES = sorted(plates)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print(len(CLASSES), "especies:", ", ".join(CLASSES))

In [ ]:
# 5. Un recorte de ejemplo por especie
import matplotlib.pyplot as plt
cols = 8
rows = -(-len(CLASSES) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.5 * rows))
for ax in axes.ravel():
    ax.axis("off")
for ax, sp in zip(axes.ravel(), CLASSES):
    files = sorted((DATASET / "train" / sp.replace(" ", "_")).glob("*.jpg"))
    if files:
        ax.imshow(cv2.cvtColor(cv2.imread(str(files[len(files) // 2])), cv2.COLOR_BGR2RGB))
    ax.set_title(sp, fontsize=8, style="italic")
plt.tight_layout(); plt.show()

In [ ]:
# 6. Entrenamiento del clasificador
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(data=str(DATASET), epochs=EPOCHS, time=TIME_HOURS, imgsz=IMGSZ, batch=BATCH, patience=10,
            workers=2, project=str(WORK / "runs"), name="species", exist_ok=True, cos_lr=True,
            fliplr=0.5, flipud=0.5, degrees=180, hsv_h=0.01, hsv_s=0.3, hsv_v=0.3)
BEST = Path(model.trainer.best)
m = YOLO(str(BEST)).val(data=str(DATASET), split="test", imgsz=IMGSZ)
print(f"TEST por colonia: top-1 = {m.top1:.3f} · top-5 = {m.top5:.3f}")

In [ ]:
# 7. Precisión por PLACA: votación entre las colonias de cada placa de test
from collections import Counter
clf = YOLO(str(BEST))
names = clf.names
ok, rows = 0, []
for sp, (path, boxes) in TEST_PLATES:
    img = cv2.imread(str(path))
    crops = [c for c in (square_crop(img, b) for b in boxes[:MAX_PER_PLATE]) if c is not None]
    probs = np.zeros(len(names))
    for i in range(0, len(crops), 128):
        for r in clf.predict(crops[i:i + 128], imgsz=IMGSZ, verbose=False):
            probs += r.probs.data.cpu().numpy()
    pred = names[int(probs.argmax())].replace("_", " ")
    ok += pred == sp
    rows.append((sp, pred, probs.max() / max(1, len(crops))))
print(f"Placas de test: {len(rows)} · especie correcta en {ok} ({100 * ok / len(rows):.1f} %)")
errors = Counter((t, p) for t, p, _ in rows if t != p)
for (t, p), n in errors.most_common(10):
    print(f"  {t} → confundida con {p} ({n} placa{'s' if n > 1 else ''})")

In [ ]:
# 8. Exportar para Bioyologic
out = WORK / "bioyologic_species.pt"
shutil.copy(BEST, out)
info = {v["species"]: {"gram": v["gram"], "agar": v["agar"]} for v in SPECIES_MAP.values()}
(WORK / "bioyologic_species.json").write_text(json.dumps({
    "pad": PAD, "imgsz": IMGSZ, "classes": [names[i].replace("_", " ") for i in range(len(names))],
    "species_info": info, "datasets": [s[0] for s in SOURCES],
}, indent=1, ensure_ascii=False))
# Un solo archivo: Chrome bloquea las descargas múltiples automáticas.
import zipfile
bundle = WORK / "bioyologic_species.zip"
with zipfile.ZipFile(bundle, "w") as z:
    z.write(out, out.name)
    z.write(WORK / "bioyologic_species.json", "bioyologic_species.json")
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print("Copia manualmente:", bundle)